##### Part 4 — Prepare the silver transaction dataset
Referring to code snippets from Python program: 
[notebooks/03_prepare_silver_data.py].

[1] - Create UNITY CATALOG volume: The path

In [0]:
# LIBRARIES
from pyspark.sql import functions as F

# SETTING CONSTANTS ACCORDING TO:
# workspace.prj_fintech-transaction-reporting-on-databricks-with-spark_sc/
catalog_name = "workspace"
schema_name = "prj_fintech-transaction-reporting-on-databricks-with-spark_sc"
volume_name = "raw_data"

raw_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"


[2] - Create UNITY CATALOG volume: The pathES to the project`s FILES

In [0]:
transactions_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/transactions.csv"
customers_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/customers.csv"
accounts_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/accounts.csv"


[3] - Read project`s FILES into SPARK DataFrames

NOTE: these have been already 
- verified in PRJ-01 Notebook!
- checked for aggregations on transactions in PRJ-02 Notebook!

In [0]:
# READ FILES INTO DATAFRAMES
transactions_df = spark.read.option("header", True).option("inferSchema", True).csv(transactions_path)
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(customers_path)
accounts_df = spark.read.option("header", True).option("inferSchema", True).csv(accounts_path)


[0] - For other Use Case: working with Temporary Views

Notes: 
- this coding could be used against SPARK DataFrames to work on Reports generated out of VIEWS, using more Business-Known syntax SQL.

- This coding is NOT ACTIVE ("commented out") !

In [0]:
# CREATE 3 RAW TEMPORARY VIEWS on top of the SPARK DataFrames:
# - raw_transactions
# - raw_customers
# - raw_accounts

#-DEACTIVATED, remove "#" transactions_df.createOrReplaceTempView("raw_transactions")
#-DEACTIVATED, remove "#" customers_df.createOrReplaceTempView("raw_customers")
#-DEACTIVATED, remove "#" accounts_df.createOrReplaceTempView("raw_accounts")

[4] - Build a prepared transaction DataFrame.

In [0]:
# Build a prepared transaction DataFrame.
# Suggested steps:
# 1. Select the needed columns.
# 2. Filter invalid rows.
# 3. Standardize status and channel values.
# 4. Create report_date.
# 5. Create amount_band.
# 6. Create is_high_value.
# 7. Join customer and account reference data.
# 8. Create is_international by comparing merchant_country and home_country.

prepared_df = (
    transactions_df
    .select(
        "transaction_id",
        "account_id",
        "customer_id",
        "transaction_type",
        "amount",
        "currency",
        "transaction_status",
        "payment_channel",
        "merchant_country",
        "transaction_date"
    )
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("account_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("transaction_date").isNotNull())
    .filter(F.col("amount") > 0)
    .withColumn("transaction_status", F.upper(F.trim(F.col("transaction_status"))))
    .withColumn("payment_channel", F.upper(F.trim(F.col("payment_channel"))))
    .withColumn("transaction_type", F.upper(F.trim(F.col("transaction_type"))))
    .withColumn("merchant_country", F.upper(F.trim(F.col("merchant_country"))))
    .withColumn("report_date", F.to_date("transaction_date"))
    .withColumn(
        "amount_band",
        F.when(F.col("amount") < 100, "LOW")
         .when(F.col("amount") < 1000, "MEDIUM")
         .otherwise("HIGH")
    )
    .withColumn("is_high_value", F.when(F.col("amount") >= 1000, 1).otherwise(0))
    .join(
        customers_df.select("customer_id", "customer_segment", "customer_country", "risk_flag"),
        on="customer_id",
        how="left"
    )
    .join(
        accounts_df.select("account_id", "account_type", "account_status", "home_country"),
        on="account_id",
        how="left"
    )
    .withColumn(
        "is_international",
        F.when(F.col("merchant_country") != F.upper(F.trim(F.col("home_country"))), 1).otherwise(0)
    )
)


display(prepared_df.limit(20))

[5] - Create a TEMPORARY VIEW on top of the "prepared transaction DataFrame".

In [0]:
prepared_df.createOrReplaceTempView("silver_transactions_view")

display(spark.sql("SELECT * FROM silver_transactions_view LIMIT 20"))

[6] - Create UNITY CATALOG table(s): The path.

In [0]:
silver_table = f"{catalog_name}.{schema_name}.silver_transactions"

print(silver_table)

[7] - Save the silver dataset as a managed UNITY CATALOG table.

Notes: 
In this cell, 1st time in the project`s Notebooks a UNITY CATALOG table is CREATED!

This notebook allows 2 options for "template" reason: 
Save the "silver dataset" into the UNITY CATALOG table 
- (A) from the TEMPORARY VIEW on top of the "prepared transaction DataFrame" (with SPARK SQL commands !); or 
- (B) from the "prepared transaction DataFrame" itself.

In [0]:
# NOTE: the actual UNITY CATALOG path with table name does not work as VARIABLE "silver_table", because the UNITY CATALOG`s SCHEMA is a mix of difficult character "-" which actually requires to receive it particularly in "''" High-Single-Quotes.   
silver_table = f"{catalog_name}.`{schema_name}`.silver_transactions"

# VARIANT (A): from the TEMPORARY VIEW on top of the "prepared transaction DataFrame" 
# (A), STEP 1: Create table with structure (columns derived from the Temporary View)
#! spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog_name}.`{schema_name}`.silver_transactions AS SELECT * FROM silver_transactions_view")

# (A), STEP 2: Fill table with records from the same Temporary View       
#! spark.sql(f"INSERT OVERWRITE {catalog_name}.`{schema_name}`.silver_transactions SELECT * FROM silver_transactions_view")

#VARIANT (B): from the "prepared transaction DataFrame" itself
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.`{schema_name}`.silver_transactions")

prepared_df.write.mode("overwrite").saveAsTable(silver_table)

# (A & B): List the records from the just created / filled table
display(spark.sql(f"SELECT * FROM {catalog_name}.`{schema_name}`.silver_transactions LIMIT 20"))

##### Project Question - Performance Thinking:
###### Q1: Where are the wider transformations in this notebook?
The biggest transformation is Building the DataFrame in one big SQL command: 
1) Left join with 2 tables (technically: all tables as DataFrames); 
2) at same moment Re-Defining a column as certain data type; 
3) at same moment Standardising multiple columns in 2 tables; 
4) and also Creating 2 additional derives columns; 
5) this all at the moment when strong Filtering is happening on the table which is driving the Join all-in-all (Left Join from it where filtering is happening). 

###### Q2: Which steps are likely to trigger shuffles?
Answering in 2 steps: 
1) All-in-all the above multiple transformation steps are hard to debug respectively to follow the correctness of the cleaning and transformation (actually that would be 2 STAGES in Pipeline Design).
2) As "Shuffle" I am considering the actual filtering on the leading DataFrame "Transactions" first, which then in the "same transformation" is going into the Left-Join with the rest of the other DataFrames (logically 2 other tables).

Conclusions: 

1) for readability and to follow a clean Data Pipeline approach, I would do these transformations on the "Leading DataFrame" in this order: 
- For Stage CLEANING:
- selection of columns to limit the size ("Column Thinking"), 
- data type casting for proper column handling, 
- standardization for proper column handling, 
- filtering (proper column handling 1st time, & "Row Thinking"); 
- For Stage BUSINESS TRANSFORMATION:
- deriving additional columns (this requires cleaned data)

2) to avoid possible suffling, I would then do the Left-Join from this cleaned and already selected & filtered DataFrame. 

